# 02: Indexing, Slicing & Reshaping (Exercises 16–30)

Deep dive into multi-dimensional indexing, strided patterns, padding, IEEE-754 semantics, and custom dtypes.

---


In [ ]:
import numpy as np
print(f"NumPy version: {np.__version__}")

### Exercise 16: How to add a border (filled with 0's) around an existing array?
**Difficulty:** `★☆☆`  
**Tags:** `Padding, Slicing`

#### 💡 Intuition & Concept
`np.pad` provides a generalized way to pad arrays along any axis. Using `mode='constant'` with `constant_values=0` adds zeros around all boundaries.

#### ⚠️ Key Takeaway & Gotchas
`np.pad` allocates a new array. If memory is tight, you can pre-allocate `np.zeros` and slice-copy the source array into the center.


In [ ]:
Z = np.ones((4, 4), dtype=int)
padded_Z = np.pad(Z, pad_width=1, mode='constant', constant_values=0)
print("Padded shape:", padded_Z.shape)
print(padded_Z)

#### 🔄 Alternative Approach / Benchmark
Manual slice assignment with pre-allocation:

In [ ]:
# Alternative via pre-allocation:
res = np.zeros((Z.shape[0] + 2, Z.shape[1] + 2), dtype=Z.dtype)
res[1:-1, 1:-1] = Z
print("Pre-allocated approach:\n", res)

### Exercise 17: What is the result of the following expressions?
**Difficulty:** `★☆☆`  
**Tags:** `NaN, Floating Point, IEEE-754`

#### 💡 Intuition & Concept
IEEE-754 floating point standard defines special values `NaN` (Not a Number) and `Inf` (Infinity). By definition, `NaN != NaN`, and any comparison with `NaN` evaluates to `False`.

#### ⚠️ Key Takeaway & Gotchas
`0.3 == 3 * 0.1` evaluates to `False` due to binary floating-point representation rounding. Always use `np.isclose` or `np.allclose` for float comparisons.


In [ ]:
print("0 * np.nan:                ", 0 * np.nan)
print("np.nan == np.nan:          ", np.nan == np.nan)
print("np.inf > np.nan:           ", np.inf > np.nan)
print("np.nan - np.nan:           ", np.nan - np.nan)
print("np.nan in set([np.nan]):   ", np.nan in set([np.nan]))
print("0.3 == 3 * 0.1:            ", 0.3 == 3 * 0.1)
print("np.isclose(0.3, 3 * 0.1):  ", np.isclose(0.3, 3 * 0.1))

### Exercise 18: Create a 5x5 matrix with values 1,2,3,4 just below the diagonal
**Difficulty:** `★☆☆`  
**Tags:** `Matrix, Diagonals`

#### 💡 Intuition & Concept
`np.diag(v, k)` constructs a diagonal matrix. When `k=0`, values are placed on the main diagonal. Negative values (`k=-1`) place values on the sub-diagonal below the main diagonal.

#### ⚠️ Key Takeaway & Gotchas
Passing a 1D vector to `np.diag` creates a 2D diagonal matrix, while passing a 2D matrix extracts its diagonal.


In [ ]:
Z = np.diag(1 + np.arange(4), k=-1)
print(Z)

### Exercise 19: Create an 8x8 matrix and fill it with a checkerboard pattern
**Difficulty:** `★☆☆`  
**Tags:** `Strided Slicing, Patterns`

#### 💡 Intuition & Concept
Step slicing with step `2` allows filling alternating cells. Setting `Z[1::2, ::2] = 1` and `Z[::2, 1::2] = 1` generates the classic alternating checkerboard.

#### ⚠️ Key Takeaway & Gotchas
Both slices update the array in-place without memory reallocations.


In [ ]:
Z = np.zeros((8, 8), dtype=int)
Z[1::2, ::2] = 1
Z[::2, 1::2] = 1
print(Z)

### Exercise 20: Consider a (6,7,8) shape array, what is the index (x,y,z) of the 100th element?
**Difficulty:** `★☆☆`  
**Tags:** `Indexing, Unravel`

#### 💡 Intuition & Concept
`np.unravel_index(flat_index, shape)` translates a 1D index into multi-dimensional coordinates based on row-major (C-style) memory ordering.

#### ⚠️ Key Takeaway & Gotchas
Remember that the 100th element corresponds to 0-based index `99`.


In [ ]:
coords = np.unravel_index(99, (6, 7, 8))
print(f"Coordinates of 100th element: {coords}")

### Exercise 21: Create a checkerboard 8x8 matrix using the tile function
**Difficulty:** `★☆☆`  
**Tags:** `Array Creation, Tiling`

#### 💡 Intuition & Concept
`np.tile(A, reps)` repeats the input array `A` the specified number of times along each axis. A base 2x2 pattern repeated (4, 4) times forms an 8x8 board.

#### ⚠️ Key Takeaway & Gotchas
`np.tile` constructs a brand new array copy, whereas stride tricks or slicing can achieve the pattern as views.


In [ ]:
base_pattern = np.array([[0, 1], [1, 0]])
checkerboard = np.tile(base_pattern, (4, 4))
print(checkerboard)

### Exercise 22: Normalize a 5x5 random matrix
**Difficulty:** `★☆☆`  
**Tags:** `Normalization, Broadcasting`

#### 💡 Intuition & Concept
Min-Max normalization scales values to $[0, 1]$ using the formula: $Z_{norm} = \frac{Z - Z_{min}}{Z_{max} - Z_{min}}$.

#### ⚠️ Key Takeaway & Gotchas
If `Z.max() == Z.min()`, division by zero will occur. In production code, add an epsilon guard.


In [ ]:
rng = np.random.default_rng(seed=42)
Z = rng.random((5, 5))
Z_norm = (Z - Z.min()) / (Z.max() - Z.min())
print("Min:", Z_norm.min(), "Max:", Z_norm.max())
print(Z_norm)

### Exercise 23: Create a custom dtype describing RGBA color (4 unsigned bytes)
**Difficulty:** `★☆☆`  
**Tags:** `Data Types, Structured Arrays`

#### 💡 Intuition & Concept
NumPy structured arrays allow defining composite record data types with named fields and specific byte representations (`np.ubyte` = `uint8`).

#### ⚠️ Key Takeaway & Gotchas
Structured dtypes are the foundation for high-performance tabular data, similar to C structs.


In [ ]:
color_dtype = np.dtype([
    ("r", np.ubyte),
    ("g", np.ubyte),
    ("b", np.ubyte),
    ("a", np.ubyte)
])
pixel = np.array((255, 128, 0, 255), dtype=color_dtype)
print(pixel)
print(f"Red: {pixel['r']}, Alpha: {pixel['a']}")

### Exercise 24: Multiply a 5x3 matrix by a 3x2 matrix (real matrix product)
**Difficulty:** `★☆☆`  
**Tags:** `Linear Algebra, Matrix Multiplication`

#### 💡 Intuition & Concept
Matrix multiplication computes dot products of rows and columns. In modern Python 3.5+, the infix `@` operator is standard.

#### ⚠️ Key Takeaway & Gotchas
The inner dimensions must match: $(M, K) \times (K, N) \to (M, N)$. Using `*` performs element-wise multiplication, not matrix multiplication!


In [ ]:
A = np.ones((5, 3))
B = np.ones((3, 2)) * 2
C = A @ B  # Modern matrix product
print("Result shape:", C.shape)
print(C)

#### 🔄 Alternative Approach / Benchmark
Using `np.dot(A, B)`:

In [ ]:
# Using np.dot:
C_dot = np.dot(A, B)
print("np.dot result:\n", C_dot)

### Exercise 25: Given a 1D array, negate all elements between 3 and 8, in place
**Difficulty:** `★☆☆`  
**Tags:** `Boolean Masking, In-place`

#### 💡 Intuition & Concept
Boolean arrays can be combined using bitwise operators (`&` for AND, `|` for OR). Using boolean indexing directly selects elements for in-place modification.

#### ⚠️ Key Takeaway & Gotchas
Do not use Python keyword `and` for element-wise boolean operations—always use `&`.


In [ ]:
Z = np.arange(11)
print("Before:", Z)
mask = (Z > 3) & (Z < 8)
Z[mask] *= -1
print("After: ", Z)

### Exercise 26: What is the output of the following script?
**Difficulty:** `★★☆`  
**Tags:** `Python Builtins, Gotchas`

#### 💡 Intuition & Concept
Python's built-in `sum(iterable, start)` takes an optional `start` value (here `-1`), so `sum(range(5), -1)` computes $0+1+2+3+4 + (-1) = 9$. Meanwhile, `np.sum(a, axis)` treats the second argument as the `axis` parameter (`axis=-1` means sum across last dimension), yielding $10$.

#### ⚠️ Key Takeaway & Gotchas
Passing parameters positionally instead of with keyword arguments (`axis=-1`) can cause subtle semantic bugs when switching between Python and NumPy functions.


In [ ]:
print("sum(range(5), -1):   ", sum(range(5), -1))
print("np.sum(range(5), -1):", np.sum(range(5), -1))

### Exercise 27: Consider an integer vector Z, which of these expressions are legal?
**Difficulty:** `★☆☆`  
**Tags:** `Syntax, Vectorization`

#### 💡 Intuition & Concept
NumPy vectorizes bit-shifts (`<<`, `>>`), complex multiplication (`1j * Z`), and arithmetic. However, chained comparison `Z < Z > Z` evaluates `(Z < Z) and (Z > Z)`, triggering a ValueError because array boolean truth values are ambiguous.

#### ⚠️ Key Takeaway & Gotchas
Always use `np.logical_and` or `&` instead of Python's chained comparison syntax with NumPy arrays.


In [ ]:
Z = np.arange(5)
print("Z**Z:         ", Z**Z)
print("2 << Z >> 2:  ", 2 << Z >> 2)
print("Z < -Z:       ", Z < -Z)
print("1j * Z:       ", 1j * Z)
print("Z / 1 / 1:    ", Z / 1 / 1)
try:
    print(Z < Z > Z)
except ValueError as e:
    print("Z < Z > Z:     Raises ValueError ->", e)

### Exercise 28: What are the results of floating point expressions?
**Difficulty:** `★☆☆`  
**Tags:** `Edge Cases, Float Division`

#### 💡 Intuition & Concept
NumPy handles division by zero using IEEE-754 standards without crashing, returning `nan` or `inf` with an optional RuntimeWarning.

#### ⚠️ Key Takeaway & Gotchas
Integer division by zero in NumPy 2.x raises a warning and returns 0.


In [ ]:
with np.errstate(divide='ignore', invalid='ignore'):
    print("np.array(0) / np.array(0):  ", np.array(0) / np.array(0))
    print("np.array(0) // np.array(0): ", np.array(0) // np.array(0))
    print("np.array([np.nan]).astype(int).astype(float):", np.array([np.nan]).astype(int).astype(float))

### Exercise 29: How to round away from zero a float array?
**Difficulty:** `★★☆`  
**Tags:** `Math, Rounding`

#### 💡 Intuition & Concept
Rounding away from zero means positive numbers round towards $+\infty$ (`ceil`) and negative numbers round towards $-\infty$ (`floor`). We can achieve this via `np.copysign(np.ceil(np.abs(Z)), Z)`.

#### ⚠️ Key Takeaway & Gotchas
Standard `np.round` rounds to the nearest even number (banker's rounding), which rounds towards zero for half-way values like $0.5$ and $-0.5$.


In [ ]:
Z = np.array([-5.5, -2.1, -0.2, 0.0, 0.2, 2.1, 5.5])
rounded = np.copysign(np.ceil(np.abs(Z)), Z)
print("Original: ", Z)
print("Away zero:", rounded)

### Exercise 30: How to find common values between two arrays?
**Difficulty:** `★☆☆`  
**Tags:** `Set Operations, Searching`

#### 💡 Intuition & Concept
`np.intersect1d(ar1, ar2)` returns the sorted, unique values that are in both input arrays.

#### ⚠️ Key Takeaway & Gotchas
It flattens multi-dimensional inputs into 1D before computing the intersection.


In [ ]:
Z1 = np.array([0, 1, 2, 3, 4, 5])
Z2 = np.array([3, 4, 5, 6, 7, 8])
common = np.intersect1d(Z1, Z2)
print("Common values:", common)